# Kilosort 4 Pipeline Notebook

This notebook breaks down the `run_kilosort` function into individual cells, allowing you to run the pipeline step-by-step.

In [1]:
%load_ext autoreload
%autoreload 2

# 1. Imports
import time
import logging
import platform
import numpy as np
import torch
from pathlib import Path

import kilosort
from kilosort import (preprocessing, datashift, template_matching, clustering_qr, 
                      io, spikedetect, CCG, PROBE_DIR)
from kilosort.run_kilosort import (
    initialize_ops, compute_preprocessing, compute_drift_correction, 
    detect_spikes, cluster_spikes, set_files, setup_logger, 
    get_run_parameters
)
from kilosort.parameters import DEFAULT_SETTINGS
from kilosort.utils import (
    log_performance, log_cuda_details, probe_as_string, ops_as_string,
    get_performance, log_sorting_summary, log_thread_count
)
import kilosort.plots as kplots

# Setup Logger
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('kilosort')

In [ ]:
# 2. Parameters & Settings
# Define your data path and probe here
data_dir = r'F:\HC_Paper\JAW_131\Home\July_30\Shank_2\Data' # UPDATE THIS PATH
filename = None
probe_path = r'F:\\HC_Paper\\JAW_131\\Channel_Map_HC.mat'

# Default settings
settings = DEFAULT_SETTINGS.copy()
settings['n_chan_bin'] = 385 # UPDATE THIS: Number of channels in binary file

settings = DEFAULT_SETTINGS
settings['n_chan_bin'] = 128
settings['fs'] = 30000
settings['probe_path']= probe_path
settings['nt']=91
settings['nt0min']=30
settings['duplicate_spike_ms']=0.8
settings['nblocks']=0
settings['dmin']=60
settings['dminx']=60
settings['whitening_range']=8
settings['nearest_chans']=4
settings['nearest_templates']=20
settings['max_channel_distance']=8
settings['tmax']=4*60*60
settings['cluster_downsampling']=1
settings['ccg_threshold']=0.1
settings['data_dir']= data_dir 

bad_channels=[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,46,64,91]

# Other parameters
data_dtype = 'int16'
do_CAR = True
invert_sign = False
save_preprocessed_copy = False
clear_cache = False
verbose_console = True
verbose_log = False
shank_idx = None
progress_bar = None
file_object = None

In [3]:
# 3. System & Device Setup
logger.info(f"Kilosort version {kilosort.__version__}")
logger.info(f"Python version {platform.python_version()}")

if torch.cuda.is_available():
    device = torch.device('cuda')
    memory = torch.cuda.get_device_properties(device).total_memory/1024**3
    logger.info(f'Using CUDA device: {torch.cuda.get_device_name()} {memory:.2f}GB')
else:
    device = torch.device('cpu')
    logger.info('Using CPU')

INFO:kilosort:Kilosort version 0.1.dev1411+ga0628b5.d20250416
INFO:kilosort:Python version 3.9.21
INFO:kilosort:Using CUDA device: NVIDIA GeForce RTX 4080 SUPER 15.99GB


In [4]:
# 4. File Setup
if not isinstance(shank_idx, list): shank_idx = [shank_idx]
# We will just run for the first shank in the list for this notebook example
idx = shank_idx[0]

_filename, _data_dir, _results_dir, _probe = \
    set_files(settings, filename, None, None, data_dir,
              None, bad_channels, idx)

setup_logger(_results_dir, verbose_console=verbose_console)
logger.info(f"Sorting {_filename}")

2026-01-31 23:18:56,469 kilosort     INFO     Sorting [WindowsPath('F:/HC_Paper/JAW_131/Home/July_30/Shank_2/Data/data.dat')]
INFO:kilosort:Sorting [WindowsPath('F:/HC_Paper/JAW_131/Home/July_30/Shank_2/Data/data.dat')]


In [5]:
# 5. Initialize Ops
tic0 = time.time()

if _probe['chanMap'].max() >= settings['n_chan_bin']:
    raise ValueError('Largest value of chanMap exceeds channel count of data.')

ops, settings = initialize_ops(
    settings, _probe, data_dtype, do_CAR, invert_sign,
    device, save_preprocessed_copy
)

logger.info(f"Initial ops:\n\n{ops_as_string(ops)}\n")
log_performance(logger, 'info', 'Resource usage before sorting')
log_thread_count(logger)

2026-01-31 23:18:56,514 kilosort     INFO     Initial ops:

ops = {
    'n_chan_bin': 128,
    'fs': 30000,
    'batch_size': 60000,
    'nblocks': 0,
    'Th_universal': 9,
    'Th_learned': 8,
    'tmin': 0,
    'tmax': 14400,
    'nt': 91,
    'shift': None,
    'scale': None,
    'batch_downsampling': 1,
    'artifact_threshold': inf,
    'nskip': 25,
    'whitening_range': 8,
    'highpass_cutoff': 300,
    'binning_depth': 5,
    'sig_interp': 20,
    'drift_smoothing': [0.5, 0.5, 0.5],
    'nt0min': 30,
    'dmin': 60,
    'dminx': 60,
    'min_template_size': 10,
    'template_sizes': 5,
    'nearest_chans': 4,
    'nearest_templates': 20,
    'max_channel_distance': 8,
    'max_peels': 100,
    'templates_from_data': True,
    'n_templates': 6,
    'n_pcs': 6,
    'Th_single_ch': 6,
    'acg_threshold': 0.2,
    'ccg_threshold': 0.1,
    'cluster_neighbors': 10,
    'cluster_downsampling': 1,
    'max_cluster_subset': 25000,
    'x_centers': None,
    'cluster_init_seed': 5,
 

In [6]:
# 6. Preprocessing (Drift Correction & Whitening)
ops = compute_preprocessing(ops, device, tic0=tic0, file_object=file_object)

# Set seeds for determinism
np.random.seed(1)
torch.cuda.manual_seed_all(1)
torch.random.manual_seed(1)

ops, bfile, st0 = compute_drift_correction(
    ops, device, tic0=tic0, progress_bar=progress_bar,
    file_object=file_object, clear_cache=clear_cache,
    verbose=verbose_log
)

log_thread_count(logger)

if save_preprocessed_copy:
    io.save_preprocessing(_results_dir / 'temp_wh.dat', ops, bfile)

# Plot drift
if st0 is not None:
    kplots.plot_drift_amount(ops, _results_dir)
    kplots.plot_drift_scatter(st0, _results_dir)

2026-01-31 23:18:56,686 kilosort.run_kilosort INFO      
INFO:kilosort.run_kilosort: 
2026-01-31 23:18:56,686 kilosort.run_kilosort INFO     Computing preprocessing variables.
INFO:kilosort.run_kilosort:Computing preprocessing variables.
2026-01-31 23:18:56,687 kilosort.run_kilosort INFO     ----------------------------------------
INFO:kilosort.run_kilosort:----------------------------------------
2026-01-31 23:18:56,710 kilosort.run_kilosort INFO     N samples: 432000000
INFO:kilosort.run_kilosort:N samples: 432000000
2026-01-31 23:18:56,710 kilosort.run_kilosort INFO     N seconds: 14400.0
INFO:kilosort.run_kilosort:N seconds: 14400.0
2026-01-31 23:18:56,711 kilosort.run_kilosort INFO     N batches: 7200
INFO:kilosort.run_kilosort:N batches: 7200
2026-01-31 23:18:59,677 kilosort.run_kilosort INFO     Preprocessing filters computed in 2.99s; total 3.16s
INFO:kilosort.run_kilosort:Preprocessing filters computed in 2.99s; total 3.16s
2026-01-31 23:18:59,678 kilosort.run_kilosort DEBUG 

In [7]:
# 7. Spike Detection (Universal Templates)
tic = time.time()
logger.info(' ')
logger.info('Spike detection')
logger.info('-'*40)

# This block corresponds to `detect_spikes` function in run_kilosort.py
st0, tF0, ops = spikedetect.run(
    ops, bfile, device=device, progress_bar=progress_bar
)
tF0 = torch.from_numpy(tF0)
elapsed = time.time() - tic
total = time.time() - tic0
ops['runtime_spikedetect'] = elapsed
ops['usage_spikedetect'] = get_performance()
logger.info(f'{len(st0)} spikes extracted in {elapsed:.2f}s; total {total:.2f}s')

if len(st0) == 0:
    raise ValueError('No spikes detected, cannot continue sorting.')

2026-01-31 23:18:59,782 kilosort     INFO      
INFO:kilosort: 
2026-01-31 23:18:59,783 kilosort     INFO     Spike detection
INFO:kilosort:Spike detection
2026-01-31 23:18:59,784 kilosort     INFO     ----------------------------------------
INFO:kilosort:----------------------------------------
2026-01-31 23:18:59,784 kilosort.spikedetect INFO     Re-computing universal templates from data.
INFO:kilosort.spikedetect:Re-computing universal templates from data.
f:\Envs\HC_KS4\lib\site-packages\threadpoolctl.py:1214: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)
2026-01-31 23:1

In [8]:
# 8. First Clustering
tic = time.time()
logger.info(' ')
logger.info('First clustering')
logger.info('-'*40)

clu, Wall = clustering_qr.run(
    ops, st0, tF0, mode='spikes', device=device, progress_bar=progress_bar,
    clear_cache=clear_cache, verbose=verbose_log
)

elapsed = time.time() - tic
logger.info(f'{clu.max()+1} clusters found in {elapsed:.2f}s')

2026-01-31 23:21:50,735 kilosort     INFO      
INFO:kilosort: 
2026-01-31 23:21:50,736 kilosort     INFO     First clustering
INFO:kilosort:First clustering
2026-01-31 23:21:50,737 kilosort     INFO     ----------------------------------------
INFO:kilosort:----------------------------------------
  0%|          | 0/1 [00:00<?, ?it/s]2026-01-31 23:21:50,808 kilosort.clustering_qr DEBUG    Center 0 | Xd shape: torch.Size([642562, 24]) | ntemp: 2
DEBUG:kilosort.clustering_qr:Center 0 | Xd shape: torch.Size([642562, 24]) | ntemp: 2
2026-01-31 23:22:01,026 kilosort.clustering_qr DEBUG    Center 1 | Xd shape: torch.Size([469941, 30]) | ntemp: 2
DEBUG:kilosort.clustering_qr:Center 1 | Xd shape: torch.Size([469941, 30]) | ntemp: 2
2026-01-31 23:22:08,924 kilosort.clustering_qr DEBUG    Center 2 | Xd shape: torch.Size([454874, 30]) | ntemp: 2
DEBUG:kilosort.clustering_qr:Center 2 | Xd shape: torch.Size([454874, 30]) | ntemp: 2
2026-01-31 23:22:16,491 kilosort.clustering_qr DEBUG    Center 3 |

In [9]:
# 9. Template Post-processing
Wall3 = template_matching.postprocess_templates(
    Wall, ops, clu, st0, tF0, device=device
)

In [10]:
# 10. Extract Spikes (Template Matching)
tic = time.time()
logger.info(' ')
logger.info('Extracting spikes using cluster waveforms')
logger.info('-'*40)

st, tF, ops = template_matching.extract(
    ops, bfile, Wall3, device=device, progress_bar=progress_bar
)

elapsed = time.time() - tic
total = time.time() - tic0
ops['runtime_st'] = elapsed
logger.info(f'{len(st)} spikes extracted in {elapsed:.2f}s; total {total:.2f}s')

2026-01-31 23:24:38,920 kilosort     INFO      
INFO:kilosort: 
2026-01-31 23:24:38,921 kilosort     INFO     Extracting spikes using cluster waveforms
INFO:kilosort:Extracting spikes using cluster waveforms
2026-01-31 23:24:38,922 kilosort     INFO     ----------------------------------------
INFO:kilosort:----------------------------------------
  0%|          | 0/7200 [00:00<?, ?it/s]2026-01-31 23:24:38,928 kilosort.template_matching DEBUG     
DEBUG:kilosort.template_matching: 
2026-01-31 23:24:38,928 kilosort.template_matching DEBUG    Batch 0
DEBUG:kilosort.template_matching:Batch 0
2026-01-31 23:24:38,929 kilosort.template_matching DEBUG    ********************************************************
DEBUG:kilosort.template_matching:********************************************************
2026-01-31 23:24:38,929 kilosort.template_matching DEBUG    CPU usage:    33.80 %
DEBUG:kilosort.template_matching:CPU usage:    33.80 %
2026-01-31 23:24:38,930 kilosort.template_matching DEBUG    

In [11]:
# 11. Final Clustering
tic = time.time()
logger.info(' ')
logger.info('Final clustering')
logger.info('-'*40)

clu, Wall = clustering_qr.run(
    ops, st, tF, mode='template', device=device, progress_bar=progress_bar,
    clear_cache=clear_cache, verbose=verbose_log
)

elapsed = time.time() - tic
logger.info(f'{clu.max()+1} clusters found in {elapsed:.2f}s')

2026-01-31 23:31:51,675 kilosort     INFO      
INFO:kilosort: 
2026-01-31 23:31:51,675 kilosort     INFO     Final clustering
INFO:kilosort:Final clustering
2026-01-31 23:31:51,676 kilosort     INFO     ----------------------------------------
INFO:kilosort:----------------------------------------
  0%|          | 0/1 [00:00<?, ?it/s]2026-01-31 23:31:51,813 kilosort.clustering_qr DEBUG    Center 0 | Xd shape: torch.Size([1208301, 24]) | ntemp: 5
DEBUG:kilosort.clustering_qr:Center 0 | Xd shape: torch.Size([1208301, 24]) | ntemp: 5
2026-01-31 23:32:18,597 kilosort.clustering_qr DEBUG    Center 1 | Xd shape: torch.Size([977303, 30]) | ntemp: 4
DEBUG:kilosort.clustering_qr:Center 1 | Xd shape: torch.Size([977303, 30]) | ntemp: 4
2026-01-31 23:32:38,700 kilosort.clustering_qr DEBUG    Center 2 | Xd shape: torch.Size([1152125, 30]) | ntemp: 8
DEBUG:kilosort.clustering_qr:Center 2 | Xd shape: torch.Size([1152125, 30]) | ntemp: 8
2026-01-31 23:33:04,138 kilosort.clustering_qr DEBUG    Center

In [12]:
# 12. Merging Clusters
tic = time.time()
logger.info(' ')
logger.info('Merging clusters')
logger.info('-'*40)

Wall_final, clu_final, is_ref_final, st_final, tF_final = template_matching.merging_function(
    ops, Wall, clu, st, tF, device=device, check_dt=True
)
clu_final = clu_final.astype('int32')

elapsed = time.time() - tic
logger.info(f'{clu_final.max()+1} units found in {elapsed:.2f}s')

2026-01-31 23:38:55,315 kilosort     INFO      
INFO:kilosort: 
2026-01-31 23:38:55,315 kilosort     INFO     Merging clusters
INFO:kilosort:Merging clusters
2026-01-31 23:38:55,316 kilosort     INFO     ----------------------------------------
INFO:kilosort:----------------------------------------
2026-01-31 23:39:07,929 kilosort     INFO     369 units found in 12.61s
INFO:kilosort:369 units found in 12.61s


In [13]:
# 13. Save Results (Phy)
# This step saves the results to disk in Phy format and performs post-processing like removing duplicates.
logger.info(f'Saving results to {_results_dir}')

io.save_to_phy(
    st_final, clu_final, tF_final, Wall_final, ops['probe'], ops, bfile.imin, results_dir=_results_dir,
    data_dtype=data_dtype, save_extra_vars=False, save_preprocessed_copy=save_preprocessed_copy
)

io.save_ops(ops, _results_dir)

logger.info('Sorting complete!')

2026-01-31 23:39:07,985 kilosort     INFO     Saving results to F:\HC_Paper\JAW_131\Home\July_30\Shank_2\Data\kilosort4
INFO:kilosort:Saving results to F:\HC_Paper\JAW_131\Home\July_30\Shank_2\Data\kilosort4
2026-01-31 23:39:07,997 kilosort.io  DEBUG     
DEBUG:kilosort.io: 
2026-01-31 23:39:07,998 kilosort.io  DEBUG    save_to_phy, whitening
DEBUG:kilosort.io:save_to_phy, whitening
2026-01-31 23:39:07,999 kilosort.io  DEBUG    ********************************************************
DEBUG:kilosort.io:********************************************************
2026-01-31 23:39:07,999 kilosort.io  DEBUG    CPU usage:    28.30 %
DEBUG:kilosort.io:CPU usage:    28.30 %
2026-01-31 23:39:08,000 kilosort.io  DEBUG    Mem used:     43.90 %     |      84.17 GB
DEBUG:kilosort.io:Mem used:     43.90 %     |      84.17 GB
2026-01-31 23:39:08,000 kilosort.io  DEBUG    Mem avail:    107.58 / 191.75 GB
DEBUG:kilosort.io:Mem avail:    107.58 / 191.75 GB
2026-01-31 23:39:08,001 kilosort.io  DEBUG    ----